---
title: "Лабораторна робота 1. Формалізація дослідницької проблеми як задачі машинного навчання та вилучення знань"
description: "Машинне навчання та методи вилучення знань | PhD F3 «Комп'ютерні науки» | КрНУ ім. М. Остроградського"
author: "&copy; [Valeriy Sydorenko](https://www.linkedin.com/in/valeriy-sydorenko-6782279a/), 2026"
date: today
jupyter: python3
bibliography: mlkdd-references.bib

format:
  html:
    code-fold: show
    code-summary: "Показати код"
    code-overflow: wrap
    toc: true
    toc-depth: 4
    toc-location: right
    number-sections: false
    embed-resources: true
    smooth-scroll: true
---

::: {.callout-note}
## Примітка

Лабораторна робота виконується після опрацювання Лекції 1 і є першим етапом *наскрізного методологічного каркаса дисертаційного ML/KDD-дослідження*.

Її результат -- не «навчена модель», а дослідницька специфікація:

$$
RQ
\rightarrow
ML/KDD\text{-задача}
\rightarrow
дані
\rightarrow
baseline
\rightarrow
експериментальний\ протокол
\rightarrow
допустиме\ наукове\ твердження.
$$

У ЛР2--ЛР8 цей артефакт послідовно уточнюватиметься.
:::

## Мета роботи

Набути практичної здатності переводити наукову проблему у формалізовану задачу машинного навчання та вилучення знань (ML/KDD); визначати одиницю аналізу, цільову популяцію, дані, змінні, тип ML/KDD-задачі, очікуваний результат і межі допустимого наукового твердження; створити початкову версію наскрізного дослідницького артефакту власної дисертаційної роботи.

## Завдання роботи

1. Сформулювати дослідницьке питання (Research Question, RQ).
2. Визначити одиницю аналізу, цільову популяцію та контекст.
3. Розрізнити наукову проблему, ML/KDD-задачу та обчислювальну процедуру.
4. Побудувати специфікацію даних і змінних.
5. Визначити тип задачі: прогнозування, опис/виявлення структури, пояснення або причинний висновок.
6. Встановити цільову величину та очікуваний тип результату.
7. Запропонувати мінімальний baseline.
8. Сформувати початковий експериментальний протокол.
9. Виконати мінімальний Python-аудит даних на навчальному прикладі.
10. Визначити межі допустимого наукового твердження.

::: {.callout-important}
## Методологічний принцип

У цій роботі **алгоритм не є початком дослідження**.

$$
\text{наукова проблема}
\rightarrow
RQ
\rightarrow
\text{дизайн}
\rightarrow
\text{дані}
\rightarrow
\text{ML/KDD-задача}
\rightarrow
\text{експеримент}
\rightarrow
\text{докази}
\rightarrow
\text{наукове твердження}.
$$

Вибір `RandomForest`, `XGBoost`, нейронної мережі чи іншого алгоритму без попередньої постановки RQ не є формалізацією наукової задачі.
:::

## Теоретичні відомості

У класичному розумінні KDD алгоритмічний пошук закономірностей є лише частиною ширшого процесу отримання валідного, нового, потенційно корисного й інтерпретованого знання [@fayyad1996kdd].

Використаємо робочу специфікацію:

$$
\mathcal{R}
=
\langle
RQ,\ U,\ P,\ D,\ X,\ Y,\ T,\ M,\ E,\ C
\rangle,
$$

де $RQ$ -- дослідницьке питання; $U$ -- одиниця аналізу; $P$ -- цільова популяція; $D$ -- дані; $X$ -- ознаки; $Y$ -- цільова величина, якщо вона є; $T$ -- тип ML/KDD-задачі; $M$ -- клас методів; $E$ -- експериментальний протокол; $C$ -- клас тверджень, які можуть підтримувати отримані докази.

## Логіка лабораторної роботи

```{mermaid}
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': '18px',
  'fontFamily': 'Trebuchet MS, Verdana, Arial, sans-serif',
  'primaryColor': '#E8F3FF',
  'primaryBorderColor': '#3478A8',
  'secondaryColor': '#FFF0C9',
  'tertiaryColor': '#E7F6EC',
  'lineColor': '#536878'
}}}%%
flowchart LR
    A("Наукова<br/>проблема") --> B("Research<br/>Question")
    B --> C("Одиниця аналізу<br/>і дані")
    C --> D("ML/KDD<br/>задача")
    D --> E("Baseline")
    E --> F("Експериментальний<br/>протокол")
    F --> G("Допустиме наукове<br/>твердження")
```

# Частина I. Навчальний кейс

Розглянемо умовне дослідження раннього виявлення ботоподібної активності в соціальних медіа. Кейс потрібний не для побудови «найкращого детектора», а для відпрацювання процедури переходу від проблеми до коректної ML/KDD-постановки.

Припустимо, маємо історичні дані про облікові записи та їхню активність. Для кожного облікового запису доступні поведінкові ознаки, обчислені лише за перші 24 години спостереження, а також незалежно сформована мітка класу.

## 1. Від теми до дослідницького питання

Невдала постановка:

> «Застосувати машинне навчання для виявлення ботів».

Це опис інструмента, а не дослідницьке питання.

Робоча постановка:

> **RQ:** якою мірою поведінкові ознаки, доступні протягом перших 24 годин активності облікового запису, дозволяють відрізняти ботоподібні облікові записи від інших у визначеній цільовій популяції?

Це RQ вже задає часову межу, одиницю аналізу, спостережувані дані, ціль і тип очікуваного висновку -- *прогнозну/дискримінаційну здатність*, а не причинність.

## 2. Одиниця аналізу та область узагальнення

| Елемент | Специфікація |
|---|---|
| Одиниця аналізу | один обліковий запис |
| Момент прогнозу | через 24 години після початку спостереження |
| Цільова популяція | облікові записи, що відповідають критеріям включення |
| Вхідна інформація | лише дані, доступні до моменту прогнозу |
| Цільова змінна | `is_bot` |
| Результат | оцінка ймовірності / прогноз класу |
| Необґрунтований перехід | «важлива для прогнозу ознака є причиною поведінки» |

::: {.callout-warning}
## Важливо

Фраза «модель працює на тестовій вибірці» не визначає область узагальнення. Потрібно вказати, для якої популяції, періоду, платформи та умов збору даних очікується застосовність результату.
:::

## 3. Від RQ до ML/KDD-задачі

Наше RQ можна операціоналізувати як задачу бінарної класифікації:

$$
f:X\rightarrow \{0,1\},
$$

або як оцінювання

$$
\hat p(y=1\mid x).
$$

Один і той самий алгоритмічний інструментарій може обслуговувати різні наукові цілі.

| Дослідницька мета | Типовий результат | Приклад твердження |
|---|---|---|
| прогноз | $\hat y$, $\hat p(y\mid x)$ | ознаки дозволяють прогнозувати клас |
| опис структури | кластери, embeddings, латентні фактори | у даних відтворюється структура |
| пояснення моделі | feature importance, локальні пояснення | модель спирається на певні ознаки |
| причинний висновок | ATE/CATE, контрфактуальний ефект | втручання змінює результат |

**Тип задачі має відповідати типу наукового питання.**

## 4. Специфікація даних до написання коду

| Змінна | Зміст | Доступна до 24 год? | Роль |
|---|---|---:|---|
| `posts_24h` | кількість повідомлень за 24 год | так | predictor |
| `night_ratio` | частка нічної активності | так | predictor |
| `unique_intervals` | різноманітність часових інтервалів | так | predictor |
| `following_growth` | приріст підписок за 24 год | так | predictor |
| `future_reports` | кількість скарг через 7 днів | **ні** | leakage |
| `is_bot` | цільова мітка | target | target |

Ознака `future_reports` може бути дуже прогнозною, але недоступна в момент раннього прогнозу. Її використання створило б *витік даних (data leakage)*:

$$
\text{predictive information}
\not\Rightarrow
\text{admissible information}.
$$

## 5. Створимо навчальні дані

Синтетичний набір дозволяє точно знати механізм генерації. Це зручно для методичного прикладу, але **не робить результати емпіричним доказом щодо реальної платформи**.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 800

is_bot = rng.binomial(1, 0.30, size=n)

posts_24h = rng.poisson(
    lam=np.where(is_bot == 1, 30, 14)
)

night_ratio = np.clip(
    rng.normal(
        loc=np.where(is_bot == 1, 0.48, 0.22),
        scale=0.12
    ),
    0, 1
)

unique_intervals = np.clip(
    rng.normal(
        loc=np.where(is_bot == 1, 4.5, 10.0),
        scale=2.0
    ),
    1, None
)

following_growth = np.clip(
    rng.normal(
        loc=np.where(is_bot == 1, 55, 18),
        scale=12
    ),
    0, None
)

# Майбутня інформація -- навмисно створена leakage-ознака.
future_reports = rng.poisson(
    lam=np.where(is_bot == 1, 8, 1)
)

df = pd.DataFrame({
    "posts_24h": posts_24h,
    "night_ratio": night_ratio,
    "unique_intervals": unique_intervals,
    "following_growth": following_growth,
    "future_reports": future_reports,
    "is_bot": is_bot
})

df.head()

,posts_24h,night_ratio,unique_intervals,following_growth,future_reports,is_bot
0,34,0.491336,8.564674,50.434807,7,1
1,19,0.221296,7.952704,14.599882,2,0
2,23,0.473227,4.906414,62.332024,4,1
3,16,0.150796,8.400685,0.000000,0,0
4,19,0.166062,8.905991,37.164334,2,0


## 6. Мінімальний аудит даних

До моделювання перевіримо форму таблиці, типи змінних, пропуски, унікальні значення та баланс цільового класу.

In [2]:
audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "n_unique": df.nunique()
})

audit

,dtype,missing,n_unique
posts_24h,int64,0,41
night_ratio,float64,0,787
unique_intervals,float64,0,795
following_growth,float64,0,761
future_reports,int64,0,19
is_bot,int64,0,2


In [3]:
class_balance = (
    df["is_bot"]
    .value_counts(normalize=True)
    .rename(index={0: "other", 1: "bot"})
    .rename("share")
)

class_balance.round(3)

is_bot
other    0.701
bot      0.299
Name: share, dtype: float64

### Інтерпретація

Ці операції ще не є ML-експериментом. Вони перевіряють, чи узгоджується обчислювальний об'єкт із дослідницькою специфікацією.

У реальному дослідженні необхідно додатково зафіксувати походження даних, процедуру формування міток, критерії включення/виключення, часовий інтервал і можливі механізми зміщення вибірки.

## 7. Визначаємо допустимий набір ознак

Сформуємо окремо інформацію, доступну в момент раннього прогнозу, та інформацію, використання якої суперечить постановці.

In [4]:
admissible_features = [
    "posts_24h",
    "night_ratio",
    "unique_intervals",
    "following_growth"
]

forbidden_features = [
    "future_reports"
]

target = "is_bot"

print("Допустимі ознаки:", admissible_features)
print("Заборонені ознаки:", forbidden_features)
print("Ціль:", target)

Допустимі ознаки: ['posts_24h', 'night_ratio', 'unique_intervals', 'following_growth']
Заборонені ознаки: ['future_reports']
Ціль: is_bot


## 8. Baseline до складної моделі

Baseline -- не «слабка модель для галочки». Він задає мінімальний рівень, з яким має сенс порівнювати складніший підхід.

Використаємо:

- `DummyClassifier` як наївний орієнтир;
- логістичну регресію як простий модельний baseline [@hastie2009elements].

Повний дизайн порівняння моделей буде предметом ЛР2.

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X = df[admissible_features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

dummy = DummyClassifier(strategy="prior")
dummy.fit(X_train, y_train)

baseline = LogisticRegression(max_iter=1000)
baseline.fit(X_train, y_train)

dummy_auc = roc_auc_score(
    y_test,
    dummy.predict_proba(X_test)[:, 1]
)

baseline_auc = roc_auc_score(
    y_test,
    baseline.predict_proba(X_test)[:, 1]
)

pd.Series({
    "Dummy ROC AUC": dummy_auc,
    "Logistic baseline ROC AUC": baseline_auc
}).round(3)

Dummy ROC AUC                0.5
Logistic baseline ROC AUC    1.0
dtype: float64

## 9. Демонстрація небезпеки leakage

Навмисно порушимо постановку й додамо майбутню ознаку `future_reports`.

Мета -- побачити, чому вища метрика може означати *гірше дослідження*.

In [6]:
leaky_features = admissible_features + ["future_reports"]

X_leaky = df[leaky_features]

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leaky,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

leaky_model = LogisticRegression(max_iter=1000)
leaky_model.fit(X_train_l, y_train_l)

leaky_auc = roc_auc_score(
    y_test_l,
    leaky_model.predict_proba(X_test_l)[:, 1]
)

pd.Series({
    "Коректний baseline ROC AUC": baseline_auc,
    "Модель із leakage ROC AUC": leaky_auc
}).round(3)

Коректний baseline ROC AUC    1.0
Модель із leakage ROC AUC     1.0
dtype: float64

### Критична інтерпретація

Якщо модель із `future_reports` має кращу метрику, це **не** робить її кращим доказом для нашого RQ. Ми поставили питання про *раннє* виявлення, але використали інформацію з майбутнього.

$$
\boxed{
\text{краща метрика}
\not\Rightarrow
\text{кращий дизайн дослідження}
}
$$

## 10. Початковий експериментальний протокол

| Компонент | Рішення |
|---|---|
| RQ | раннє розрізнення ботоподібних та інших облікових записів |
| Одиниця аналізу | обліковий запис |
| Горизонт спостереження | перші 24 години |
| Target | `is_bot` |
| Ознаки | лише доступні до моменту прогнозу |
| Baseline 0 | `DummyClassifier` |
| Baseline 1 | логістична регресія |
| Первинна метрика | ROC AUC |
| Split | stratified train/test |
| Заборонена інформація | post-outcome / future features |
| Науковий результат | оцінка прогнозної інформативності ознак |
| Необґрунтований результат | причинна інтерпретація ознак |

Це версія 0.1. Temporal split, повторна валідація, невизначеність, зовнішня валідність і distribution shift будуть системно опрацьовані далі.

## 11. Критичний аудит навчального кейсу

| Питання аудиту | Відповідь здобувача |
|---|---|
| Яке саме RQ ми поставили? | |
| Що є одиницею аналізу? | |
| На яку популяцію хочемо узагальнювати? | |
| Яка інформація доступна в момент прогнозу? | |
| Яка ознака створює leakage і чому? | |
| Що саме вимірює ROC AUC? | |
| Чому висока ROC AUC не доводить причинність? | |
| Який baseline використано і навіщо? | |
| Які загрози валідності ще не перевірено? | |
| Яке найсильніше коректне твердження можна сформулювати? | |
| Яке твердження було б надмірним? | |

::: {.callout-tip}
## Критерій якості

Сильна відповідь чітко встановлює відповідність:

$$
RQ
\leftrightarrow
дані
\leftrightarrow
ML/KDD\text{-задача}
\leftrightarrow
експеримент
\leftrightarrow
тип\ висновку.
$$
:::

# Частина II. Перенесення на дисертаційне дослідження

Повторіть процедуру для власного дослідження.

На ЛР1 не потрібно будувати остаточну модель або мати завершений датасет. Потрібно створити *дослідницьку специфікацію v0.1*, достатньо чітку, щоб у ЛР2 перейти до baseline та повноцінного ML-експерименту.

## 12. Наукова проблема та RQ

**Проблема:** ...

**RQ:** ...

Перевірте, чи RQ:

- вказує на досліджуваний об'єкт/процес;
- визначає властивість, залежність, структуру або ефект;
- допускає емпіричну перевірку;
- не підмінене назвою алгоритму;
- узгоджується з доступними або потенційно доступними даними.

## 13. Паспорт ML/KDD-дослідження v0.1

| Елемент | Специфікація |
|---|---|
| Наукова проблема | |
| Research Question | |
| Одиниця аналізу | |
| Цільова популяція / область узагальнення | |
| Контекст і часові межі | |
| Джерело даних | |
| Спосіб формування вибірки | |
| Основні змінні / представлення | |
| Target / outcome, якщо є | |
| Тип ML/KDD-задачі | |
| Очікуваний результат | |
| Мінімальний baseline | |
| Основна метрика / критерій | |
| Потенційні leakage-ознаки | |
| Основні джерела невизначеності | |
| Потенційні загрози валідності | |
| Найсильніше допустиме наукове твердження | |
| Твердження, яке ці дані не дозволяють робити | |

## 14. Концептуальна схема власного дослідження

Замініть приклад власною схемою, що відображає *дослідницьку*, а не програмну архітектуру.

```{mermaid}
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': '18px',
  'fontFamily': 'Trebuchet MS, Verdana, Arial, sans-serif',
  'primaryColor': '#E8F3FF',
  'primaryBorderColor': '#3478A8',
  'secondaryColor': '#FFF0C9',
  'tertiaryColor': '#E7F6EC',
  'lineColor': '#536878'
}}}%%
flowchart LR
    A("Research<br/>Question") --> B("Популяція<br/>та вибірка")
    B --> C("Дані та<br/>представлення")
    C --> D("ML/KDD<br/>задача")
    D --> E("Експеримент")
    E --> F("Докази")
    F --> G("Наукове<br/>твердження")
```

## 15. Матриця «RQ -- задача -- результат -- твердження»

| RQ / підпитання | ML/KDD-задача | Обчислювальний результат | Що він може підтримувати | Чого він не доводить |
|---|---|---|---|---|
| ... | ... | ... | ... | ... |
| ... | ... | ... | ... | ... |

Якщо дисертація поєднує класифікацію, кластеризацію, XAI та причинний аналіз, заповніть окремий рядок для кожного класу задач.

## 16. Специфікація даних

| Змінна / поле | Семантика | Джерело | Момент доступності | Роль | Ризик leakage / bias |
|---|---|---|---|---|---|
| ... | ... | ... | ... | ... | ... |

Якщо даних ще немає, таблиця має описувати *планований* набір даних.

## 17. Мінімальний аудит власних даних

Якщо дані вже доступні, виконайте мінімальний аудит у Python. Якщо дані ще збираються, адаптуйте каркас до очікуваної схеми.

In [ ]:
# Приклад каркаса. Замініть на власні дані.

# data = pd.read_csv("your_data.csv")

# print(data.shape)
# display(data.head())
# display(data.dtypes)
# display(data.isna().sum().sort_values(ascending=False).head(20))
# print("Кількість повних дублікатів:", data.duplicated().sum())

# Якщо є target:
# display(data["target"].value_counts(dropna=False, normalize=True))

## 18. Baseline і первинний критерій порівняння

Запропонуйте *мінімально достатній* baseline.

- **Baseline 0:** ...
- **Baseline 1:** ...
- **Запропонований клас моделей:** ...
- **Первинний критерій порівняння:** ...
- **Чому цей критерій відповідає RQ:** ...

Якщо дослідження не є supervised ML, визначте аналог baseline: випадкову структуру, просту евристику, null model, альтернативну метрику або простіший метод.

## 19. Експериментальний протокол v0.1

| Елемент | Рішення |
|---|---|
| Дані для навчання / побудови закономірності | |
| Дані для оцінювання | |
| Принцип поділу даних | |
| Baseline | |
| Основний метод | |
| Первинна метрика / критерій | |
| Додаткові метрики | |
| Контроль leakage | |
| Random seed / повтори | |
| Що буде зафіксовано до експерименту | |
| Що може змінюватися після первинного результату | |
| Який результат вважатиметься негативним | |

Останній рядок принциповий: науковий дизайн має допускати результат, за якого гіпотеза або запропонований метод *не отримує підтримки*.

## 20. Межі наукового твердження

Сформулюйте три речення.

**1. Якщо експеримент дасть очікуваний результат, я зможу стверджувати:**

> ...

**2. Я не зможу на підставі цього експерименту стверджувати:**

> ...

**3. Для сильнішого твердження додатково потрібні:**

> ...

Ця вправа захищає від переходу

$$
\text{model output}
\rightarrow
\text{надмірна інтерпретація}
\rightarrow
\text{необґрунтоване наукове твердження}.
$$

## 21. Підсумковий артефакт ЛР1

Після виконання роботи має бути сформована версія **ML/KDD Research Specification v0.1**:

```{mermaid}
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': '18px',
  'fontFamily': 'Trebuchet MS, Verdana, Arial, sans-serif',
  'primaryColor': '#E8F3FF',
  'primaryBorderColor': '#3478A8',
  'secondaryColor': '#FFF0C9',
  'tertiaryColor': '#E7F6EC',
  'lineColor': '#536878'
}}}%%
flowchart LR
    RQ("Research<br/>Question") --> DS("Data<br/>Specification")
    DS --> TK("ML/KDD<br/>Task")
    TK --> BL("Baseline")
    BL --> EP("Experimental<br/>Protocol v0.1")
    EP --> CL("Claim<br/>Boundary")
```

У ЛР2 цей артефакт буде розширено до:

**Research Specification v0.1 → Baseline Model → ML Experiment Design v0.2.**

## Контрольні питання

1. Чим наукова проблема відрізняється від ML/KDD-задачі?
2. Чому формулювання «застосувати нейронну мережу» не є дослідницьким питанням?
3. Що таке одиниця аналізу і чому її потрібно визначити до моделювання?
4. Чим цільова вибірка відрізняється від цільової популяції?
5. Як момент доступності ознаки пов'язаний із data leakage?
6. Чому висока прогнозна метрика не гарантує валідності дослідження?
7. Для чого потрібен baseline?
8. Чим прогнозне твердження відрізняється від причинного?
9. Чому XAI-пояснення не перетворює прогнозну модель на причинну?
10. Які ML/KDD-задачі можуть не мати цільової змінної $Y$?
11. Що має містити початковий експериментальний протокол?
12. Навіщо заздалегідь формулювати межу допустимого наукового твердження?
13. Який зв'язок між RQ, даними, метрикою та типом висновку?
14. Чому негативний результат має бути допустимим результатом наукового експерименту?
15. Які елементи специфікації ЛР1 потребуватимуть уточнення у наступних роботах?

## Вимоги до звіту

Звіт формується на основі виконаного зошита і має містити:

1. наукову проблему та одне основне RQ;
2. паспорт ML/KDD-дослідження v0.1;
3. концептуальну схему;
4. матрицю «RQ -- задача -- результат -- твердження»;
5. специфікацію даних;
6. результати мінімального аудиту даних або план такого аудиту;
7. baseline та обґрунтування первинної метрики;
8. експериментальний протокол v0.1;
9. межі допустимого наукового твердження;
10. короткий критичний висновок.

Код, таблиці та графіки мають бути відтворюваними.

## Висновки

ЛР1 фіксує головний принцип практикуму:

$$
\boxed{
\text{спочатку дослідницька постановка -- потім алгоритм}
}
$$

Якісний ML/KDD-експеримент починається з узгодження *RQ, одиниці аналізу, даних, типу задачі, baseline, протоколу оцінювання та класу допустимих тверджень*.

Створена **ML/KDD Research Specification v0.1** є першим артефактом наскрізного дисертаційного дослідження.

## Джерела та корисні ресурси

Основні концептуальні положення спираються на класичне розуміння KDD [@fayyad1996kdd], постановку машинного навчання як навчання за даними [@mitchell1997ml], статистичне навчання [@hastie2009elements] і розрізнення прогнозної та пояснювальної логіки моделювання [@breiman2001two].

## References

::: {#refs}
:::